# 01 — Direct VQA evaluation

This notebook runs **Qwen3-VL-4B** on a 100-question subset of the DeepTumorVQA
benchmark in direct (non-agent) mode using 2D image input + multiple-choice
scoring, then prints the model's rank on the paper leaderboard.

Expected wall time: ~3 min on a single A5000 (after the model is downloaded).


## 1. Install


In [ ]:
!pip install -q deeptumorvqa[hf]


## 2. Run the evaluator

The CLI fetches only the artifacts it needs (~700 MB of 2D PNGs), not the
30 GB of CT volumes.


In [ ]:
!deeptumorvqa-eval \
    --model Qwen/Qwen3-VL-4B-Instruct \
    --backend hf \
    --mode vqa --input 2d_image --format mc \
    --limit 100 \
    --output results/qwen3vl_smoke.json \
    --label "Qwen3-VL-4B (smoke)"


## 3. Inspect the results

The JSON output contains both the per-question predictions and the aggregated
metrics. Free-form mode is the same call with `--format freeform`.


In [ ]:
import json
data = json.load(open("results/qwen3vl_smoke.json"))
print("Overall:", data["metrics"]["overall"])
print("\nPer super-type:")
for st, m in data["metrics"]["by_super_type"].items():
    print(f"  {st:25s}  {m['accuracy']*100:>5.1f}%  (N={m['n_total']})")


## 4. What happens on the full 10K benchmark

Drop `--limit 100` and re-run. Expected paper number for Qwen3-VL-4B
direct VQA / 2D image / MC: **37.8% Overall**. A 100-sample smoke gives
±5 pp 95% CI so anything in the 33-43% range is consistent.

Switch the model with `--model <hf_id_or_local_path>`. The reference impl
supports any Qwen3-VL family model via vLLM (`--backend vllm`) or HF
transformers (`--backend hf`). For other model families, see notebook 03.
